# Credit Risk Default Prediction

Dataset: Kaggle "Give Me Some Credit"

## Step 1: Load data, explore missingness and class imbalance

In [ ]:
import pandas as pd
import numpy as np

data_dir = r'C:\Users\User\Downloads\credit-risk-project\data'

train = pd.read_csv(data_dir + r'\cs-training.csv', index_col=0)
test = pd.read_csv(data_dir + r'\cs-test.csv', index_col=0)

print(train.shape, test.shape)
train.head()

In [ ]:
train.isnull().sum()

In [ ]:
train['SeriousDlqin2yrs'].value_counts(normalize=True)

In [ ]:
train['NumberOfDependents'].value_counts(normalize=True).sort_index()

In [ ]:
from sklearn.model_selection import train_test_split

X = train.drop(columns='SeriousDlqin2yrs')
y = train['SeriousDlqin2yrs']

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(X_train.shape, X_val.shape)
print(y_train.mean(), y_val.mean())

In [ ]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy='median')

X_train_imputed = pd.DataFrame(
    imputer.fit_transform(X_train), columns=X_train.columns, index=X_train.index
)
X_val_imputed = pd.DataFrame(
    imputer.transform(X_val), columns=X_val.columns, index=X_val.index
)

X_train_imputed.isnull().sum().sum(), X_val_imputed.isnull().sum().sum()

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train_imputed), columns=X_train_imputed.columns, index=X_train_imputed.index
)
X_val_scaled = pd.DataFrame(
    scaler.transform(X_val_imputed), columns=X_val_imputed.columns, index=X_val_imputed.index
)

X_train_scaled.describe().loc[['mean', 'std']]

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

model = LogisticRegression(random_state=42)
model.fit(X_train_scaled, y_train)

val_proba = model.predict_proba(X_val_scaled)[:, 1]
roc_auc_score(y_val, val_proba)

In [ ]:
from sklearn.metrics import classification_report

val_pred = model.predict(X_val_scaled)
print(classification_report(y_val, val_pred))

In [ ]:
threshold = 0.2
val_pred_thresh = (val_proba >= threshold).astype(int)
print(classification_report(y_val, val_pred_thresh))

In [ ]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(random_state=42)
xgb_model.fit(X_train_imputed, y_train)

xgb_val_proba = xgb_model.predict_proba(X_val_imputed)[:, 1]
roc_auc_score(y_val, xgb_val_proba)

In [ ]:
xgb_val_pred = xgb_model.predict(X_val_imputed)
print(classification_report(y_val, xgb_val_pred))

In [ ]:
from sklearn.metrics import precision_recall_curve

precisions, recalls, thresholds = precision_recall_curve(y_val, xgb_val_proba)

f1_scores = 2 * precisions * recalls / (precisions + recalls + 1e-12)
f1_scores = f1_scores[:-1]  # last precision/recall point has no corresponding threshold

best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

print(f'best threshold: {best_threshold:.4f}')
print(f'precision: {precisions[best_idx]:.4f}, recall: {recalls[best_idx]:.4f}, f1: {f1_scores[best_idx]:.4f}')

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 6))
plt.plot(recalls, precisions, label='Precision-Recall curve')
plt.scatter(recalls[best_idx], precisions[best_idx], color='red', zorder=5,
            label=f'Best F1 = {f1_scores[best_idx]:.2f} (threshold={best_threshold:.2f})')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve (XGBoost)')
plt.legend()

# zoom in around the best F1 point -- comment these two lines out to see the full curve
plt.xlim(recalls[best_idx] - 0.2, recalls[best_idx] + 0.2)
plt.ylim(precisions[best_idx] - 0.2, precisions[best_idx] + 0.2)

plt.show()

In [ ]:
from sklearn.metrics import roc_curve

fpr_lr, tpr_lr, _ = roc_curve(y_val, val_proba)
fpr_xgb, tpr_xgb, _ = roc_curve(y_val, xgb_val_proba)

plt.figure(figsize=(6, 6))
plt.plot(fpr_lr, tpr_lr, label=f'Logistic Regression (AUC={roc_auc_score(y_val, val_proba):.3f})')
plt.plot(fpr_xgb, tpr_xgb, label=f'XGBoost (AUC={roc_auc_score(y_val, xgb_val_proba):.3f})')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Random guess')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve Comparison')
plt.legend()
plt.show()

In [ ]:
lr_precisions, lr_recalls, lr_thresholds = precision_recall_curve(y_val, val_proba)
lr_f1_scores = 2 * lr_precisions * lr_recalls / (lr_precisions + lr_recalls + 1e-12)
lr_f1_scores = lr_f1_scores[:-1]
lr_best_idx = np.argmax(lr_f1_scores)
lr_best_threshold = lr_thresholds[lr_best_idx]

plt.figure(figsize=(6, 6))
plt.plot(lr_recalls, lr_precisions, label='Precision-Recall curve')
plt.scatter(lr_recalls[lr_best_idx], lr_precisions[lr_best_idx], color='red', zorder=5,
            label=f'Best F1 = {lr_f1_scores[lr_best_idx]:.2f} (threshold={lr_best_threshold:.2f})')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve (Logistic Regression)')
plt.legend()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(6, 10), sharex=True)

axes[0].plot(recalls, precisions, label='Precision-Recall curve')
axes[0].scatter(recalls[best_idx], precisions[best_idx], color='red', zorder=5,
                label=f'Best F1 = {f1_scores[best_idx]:.2f} (threshold={best_threshold:.2f})')
axes[0].set_ylabel('Precision')
axes[0].set_title('XGBoost')
axes[0].legend()

axes[1].plot(lr_recalls, lr_precisions, label='Precision-Recall curve')
axes[1].scatter(lr_recalls[lr_best_idx], lr_precisions[lr_best_idx], color='red', zorder=5,
                label=f'Best F1 = {lr_f1_scores[lr_best_idx]:.2f} (threshold={lr_best_threshold:.2f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Logistic Regression')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
importances = pd.Series(xgb_model.feature_importances_, index=X_train_imputed.columns).sort_values(ascending=False)

plt.figure(figsize=(8, 5))
importances.plot(kind='barh')
plt.gca().invert_yaxis()
plt.xlabel('Feature Importance (gain)')
plt.title('XGBoost Feature Importance')
plt.show()

importances

In [ ]:
print(X_train['RevolvingUtilizationOfUnsecuredLines'].describe())
print()
print('rows with utilization > 1:', (X_train['RevolvingUtilizationOfUnsecuredLines'] > 1).sum())
print('rows with utilization > 10:', (X_train['RevolvingUtilizationOfUnsecuredLines'] > 10).sum())

In [ ]:
cap = X_train['RevolvingUtilizationOfUnsecuredLines'].quantile(0.99)
print('cap value:', cap)

X_train_capped = X_train.copy()
X_val_capped = X_val.copy()
X_train_capped['RevolvingUtilizationOfUnsecuredLines'] = X_train_capped['RevolvingUtilizationOfUnsecuredLines'].clip(upper=cap)
X_val_capped['RevolvingUtilizationOfUnsecuredLines'] = X_val_capped['RevolvingUtilizationOfUnsecuredLines'].clip(upper=cap)

imputer_capped = SimpleImputer(strategy='median')
X_train_capped_imputed = pd.DataFrame(
    imputer_capped.fit_transform(X_train_capped), columns=X_train_capped.columns, index=X_train_capped.index
)
X_val_capped_imputed = pd.DataFrame(
    imputer_capped.transform(X_val_capped), columns=X_val_capped.columns, index=X_val_capped.index
)

scaler_capped = StandardScaler()
X_train_capped_scaled = pd.DataFrame(
    scaler_capped.fit_transform(X_train_capped_imputed), columns=X_train_capped_imputed.columns, index=X_train_capped_imputed.index
)
X_val_capped_scaled = pd.DataFrame(
    scaler_capped.transform(X_val_capped_imputed), columns=X_val_capped_imputed.columns, index=X_val_capped_imputed.index
)

model_capped = LogisticRegression(random_state=42)
model_capped.fit(X_train_capped_scaled, y_train)

val_proba_capped = model_capped.predict_proba(X_val_capped_scaled)[:, 1]
roc_auc_score(y_val, val_proba_capped)

In [ ]:
print(X_train['DebtRatio'].describe())
print()
print('rows with DebtRatio > 1:', (X_train['DebtRatio'] > 1).sum())
print('rows with DebtRatio > 10:', (X_train['DebtRatio'] > 10).sum())

In [ ]:
high_debt_ratio = X_train['DebtRatio'] > 10

print('MonthlyIncome missing rate when DebtRatio > 10:', X_train.loc[high_debt_ratio, 'MonthlyIncome'].isnull().mean())
print('MonthlyIncome missing rate when DebtRatio <= 10:', X_train.loc[~high_debt_ratio, 'MonthlyIncome'].isnull().mean())

In [ ]:
X_train_v2 = X_train_capped.copy()
X_val_v2 = X_val_capped.copy()

X_train_v2['MonthlyIncome_was_missing'] = X_train['MonthlyIncome'].isnull().astype(int)
X_val_v2['MonthlyIncome_was_missing'] = X_val['MonthlyIncome'].isnull().astype(int)

X_train_v2.loc[X_train_v2['MonthlyIncome_was_missing'] == 1, 'DebtRatio'] = np.nan
X_val_v2.loc[X_val_v2['MonthlyIncome_was_missing'] == 1, 'DebtRatio'] = np.nan

imputer_v2 = SimpleImputer(strategy='median')
X_train_v2_imputed = pd.DataFrame(
    imputer_v2.fit_transform(X_train_v2), columns=X_train_v2.columns, index=X_train_v2.index
)
X_val_v2_imputed = pd.DataFrame(
    imputer_v2.transform(X_val_v2), columns=X_val_v2.columns, index=X_val_v2.index
)

scaler_v2 = StandardScaler()
X_train_v2_scaled = pd.DataFrame(
    scaler_v2.fit_transform(X_train_v2_imputed), columns=X_train_v2_imputed.columns, index=X_train_v2_imputed.index
)
X_val_v2_scaled = pd.DataFrame(
    scaler_v2.transform(X_val_v2_imputed), columns=X_val_v2_imputed.columns, index=X_val_v2_imputed.index
)

model_v2 = LogisticRegression(random_state=42)
model_v2.fit(X_train_v2_scaled, y_train)

val_proba_v2 = model_v2.predict_proba(X_val_v2_scaled)[:, 1]
roc_auc_score(y_val, val_proba_v2)

In [ ]:
X_train_v3 = X_train_capped.copy()
X_val_v3 = X_val_capped.copy()

X_train_v3['MonthlyIncome_was_missing'] = X_train['MonthlyIncome'].isnull().astype(int)
X_val_v3['MonthlyIncome_was_missing'] = X_val['MonthlyIncome'].isnull().astype(int)

mask_train = X_train_v3['MonthlyIncome_was_missing'] == 1
mask_val = X_val_v3['MonthlyIncome_was_missing'] == 1

# recover a per-row ratio using the raw stored value (assumed to be $ debt) / imputed income
X_train_v3.loc[mask_train, 'DebtRatio'] = X_train_v3.loc[mask_train, 'DebtRatio'] / X_train_capped_imputed.loc[mask_train, 'MonthlyIncome']
X_val_v3.loc[mask_val, 'DebtRatio'] = X_val_v3.loc[mask_val, 'DebtRatio'] / X_val_capped_imputed.loc[mask_val, 'MonthlyIncome']

imputer_v3 = SimpleImputer(strategy='median')
X_train_v3_imputed = pd.DataFrame(
    imputer_v3.fit_transform(X_train_v3), columns=X_train_v3.columns, index=X_train_v3.index
)
X_val_v3_imputed = pd.DataFrame(
    imputer_v3.transform(X_val_v3), columns=X_val_v3.columns, index=X_val_v3.index
)

scaler_v3 = StandardScaler()
X_train_v3_scaled = pd.DataFrame(
    scaler_v3.fit_transform(X_train_v3_imputed), columns=X_train_v3_imputed.columns, index=X_train_v3_imputed.index
)
X_val_v3_scaled = pd.DataFrame(
    scaler_v3.transform(X_val_v3_imputed), columns=X_val_v3_imputed.columns, index=X_val_v3_imputed.index
)

model_v3 = LogisticRegression(random_state=42)
model_v3.fit(X_train_v3_scaled, y_train)

val_proba_v3 = model_v3.predict_proba(X_val_v3_scaled)[:, 1]
roc_auc_score(y_val, val_proba_v3)

In [ ]:
pd.Series(model_v3.coef_[0], index=X_train_v3_scaled.columns).sort_values(key=abs, ascending=False)

In [ ]:
late_payment_cols = ['NumberOfTime30-59DaysPastDueNotWorse', 'NumberOfTime60-89DaysPastDueNotWorse', 'NumberOfTimes90DaysLate']
X_train[late_payment_cols].corr()

In [ ]:
print(X_train[late_payment_cols].describe())
print()
for col in late_payment_cols:
    print(col)
    print(X_train[col].value_counts().sort_index(ascending=False).head(5))
    print()

In [ ]:
X_train_v4 = X_train_v3.copy()
X_val_v4 = X_val_v3.copy()

sentinel_mask_train = (X_train[late_payment_cols] >= 96).any(axis=1)
sentinel_mask_val = (X_val[late_payment_cols] >= 96).any(axis=1)

X_train_v4['HasSentinelLatePaymentValue'] = sentinel_mask_train.astype(int)
X_val_v4['HasSentinelLatePaymentValue'] = sentinel_mask_val.astype(int)

X_train_v4.loc[sentinel_mask_train, late_payment_cols] = np.nan
X_val_v4.loc[sentinel_mask_val, late_payment_cols] = np.nan

imputer_v4 = SimpleImputer(strategy='median')
X_train_v4_imputed = pd.DataFrame(
    imputer_v4.fit_transform(X_train_v4), columns=X_train_v4.columns, index=X_train_v4.index
)
X_val_v4_imputed = pd.DataFrame(
    imputer_v4.transform(X_val_v4), columns=X_val_v4.columns, index=X_val_v4.index
)

scaler_v4 = StandardScaler()
X_train_v4_scaled = pd.DataFrame(
    scaler_v4.fit_transform(X_train_v4_imputed), columns=X_train_v4_imputed.columns, index=X_train_v4_imputed.index
)
X_val_v4_scaled = pd.DataFrame(
    scaler_v4.transform(X_val_v4_imputed), columns=X_val_v4_imputed.columns, index=X_val_v4_imputed.index
)

model_v4 = LogisticRegression(random_state=42)
model_v4.fit(X_train_v4_scaled, y_train)

val_proba_v4 = model_v4.predict_proba(X_val_v4_scaled)[:, 1]
roc_auc_score(y_val, val_proba_v4)

In [ ]:
pd.Series(model_v4.coef_[0], index=X_train_v4_scaled.columns).sort_values(key=abs, ascending=False)

In [ ]:
X_train_v5 = X_train_v4_imputed.drop(columns=['HasSentinelLatePaymentValue', 'MonthlyIncome_was_missing'])
X_val_v5 = X_val_v4_imputed.drop(columns=['HasSentinelLatePaymentValue', 'MonthlyIncome_was_missing'])

scaler_v5 = StandardScaler()
X_train_v5_scaled = pd.DataFrame(
    scaler_v5.fit_transform(X_train_v5), columns=X_train_v5.columns, index=X_train_v5.index
)
X_val_v5_scaled = pd.DataFrame(
    scaler_v5.transform(X_val_v5), columns=X_val_v5.columns, index=X_val_v5.index
)

model_v5 = LogisticRegression(random_state=42)
model_v5.fit(X_train_v5_scaled, y_train)

val_proba_v5 = model_v5.predict_proba(X_val_v5_scaled)[:, 1]
roc_auc_score(y_val, val_proba_v5)

In [ ]:
xgb_model_v5 = XGBClassifier(random_state=42)
xgb_model_v5.fit(X_train_v5, y_train)

xgb_val_proba_v5 = xgb_model_v5.predict_proba(X_val_v5)[:, 1]
roc_auc_score(y_val, xgb_val_proba_v5)

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline

LATE_PAYMENT_COLS = ['NumberOfTime30-59DaysPastDueNotWorse', 'NumberOfTime60-89DaysPastDueNotWorse', 'NumberOfTimes90DaysLate']

class CreditDataCleaner(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self.utilization_cap_ = X['RevolvingUtilizationOfUnsecuredLines'].quantile(0.99)
        self.income_median_ = X['MonthlyIncome'].median()
        return self

    def transform(self, X):
        X = X.copy()

        X['RevolvingUtilizationOfUnsecuredLines'] = X['RevolvingUtilizationOfUnsecuredLines'].clip(upper=self.utilization_cap_)

        income_missing = X['MonthlyIncome'].isnull()
        X.loc[income_missing, 'DebtRatio'] = X.loc[income_missing, 'DebtRatio'] / self.income_median_

        sentinel_mask = (X[LATE_PAYMENT_COLS] >= 96).any(axis=1)
        X.loc[sentinel_mask, LATE_PAYMENT_COLS] = np.nan

        return X

In [ ]:
logreg_pipeline = Pipeline([
    ('cleaner', CreditDataCleaner()),
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(random_state=42))
])

logreg_pipeline.fit(X_train, y_train)

pipeline_val_proba = logreg_pipeline.predict_proba(X_val)[:, 1]
roc_auc_score(y_val, pipeline_val_proba)

In [ ]:
from sklearn.metrics import confusion_matrix

FN_COST = 5  # assumed relative cost of missing an actual defaulter (lost exposure on a bad loan)
FP_COST = 1  # assumed relative cost of unnecessarily flagging a good customer (lost marginal profit)

candidate_thresholds = np.linspace(0.01, 0.99, 99)
total_costs = []

for t in candidate_thresholds:
    y_pred = (xgb_val_proba >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_val, y_pred).ravel()
    total_costs.append(FN_COST * fn + FP_COST * fp)

total_costs = np.array(total_costs)
best_idx = np.argmin(total_costs)
best_cost_threshold = candidate_thresholds[best_idx]

# for comparison: cost at the default 0.5 threshold
default_pred = (xgb_val_proba >= 0.5).astype(int)
tn, fp, fn, tp = confusion_matrix(y_val, default_pred).ravel()
cost_at_default = FN_COST * fn + FP_COST * fp

print(f'best threshold: {best_cost_threshold:.3f}')
print(f'total cost at best threshold: {total_costs[best_idx]:.0f}')
print(f'total cost at default 0.5 threshold: {cost_at_default:.0f}')

plt.figure(figsize=(8, 5))
plt.plot(candidate_thresholds, total_costs)
plt.scatter(best_cost_threshold, total_costs[best_idx], color='red', zorder=5,
            label=f'Min cost at threshold={best_cost_threshold:.2f}')
plt.xlabel('Threshold')
plt.ylabel(f'Total cost (FN_cost={FN_COST}, FP_cost={FP_COST})')
plt.title('Expected Cost vs. Threshold (XGBoost)')
plt.legend()
plt.show()

## Summary

**Goal**: predict probability of serious delinquency within 2 years using the "Give Me Some Credit" dataset (150,000 borrowers, ~6.7% default rate).

**Methodology**
1. Explored missingness (`MonthlyIncome`: 19.8%, `NumberOfDependents`: 2.6%) and confirmed severe class imbalance, ruling out accuracy as an evaluation metric.
2. Split off a stratified validation set from `cs-training.csv` (Kaggle's `cs-test.csv` has no usable labels — confirmed early on).
3. Built a `SimpleImputer` (median) → `StandardScaler` → `LogisticRegression` baseline: **AUC = 0.714**.
4. Built an `XGBClassifier` baseline (median-imputed only, no scaling needed): **AUC = 0.860** — a large gap over logistic regression, consistent with XGBoost's ability to capture non-linear feature interactions.
5. Investigated *why* the gap was so large, rather than accepting it:
   - `RevolvingUtilizationOfUnsecuredLines` had extreme outliers (max 50,708) distorting `StandardScaler`'s mean/std. Capping at the train-derived 99th percentile alone brought logistic regression to **AUC = 0.811**.
   - `DebtRatio`'s extreme values (max 329,664) were shown to correlate almost perfectly (92.6%) with missing `MonthlyIncome` — strong evidence the raw debt amount was stored in place of a ratio when income was unreported. Fixing this added a `MonthlyIncome_was_missing` flag and recovered a per-row ratio estimate — but the coefficient on `DebtRatio` turned out to be negligible either way (**AUC ≈ 0.812**), showing not every diagnosed data issue is worth fixing for its own sake.
   - The three late-payment count columns showed suspiciously high pairwise correlation (0.98–0.99), traced to 214 rows sharing an identical sentinel value (98 or 96) across all three columns — a data-encoding artifact, not real payment history. Flagging and nulling these rows fixed both a counterintuitive negative coefficient (multicollinearity-driven sign flip) and pushed logistic regression to **AUC = 0.856** — nearly matching XGBoost's 0.860 through data cleaning alone, no hyperparameter tuning.
6. Wrapped the full cleaning → impute → scale → model sequence into a single `sklearn.Pipeline` with a custom `CreditDataCleaner` transformer, enforcing the fit-on-train-only discipline structurally rather than manually.
7. Explored threshold selection beyond the default 0.5: F1-maximizing threshold (found by direct search over `precision_recall_curve`, since precision/recall are step functions of a finite sample — not differentiable), and a cost-based threshold that explicitly weights missing a defaulter more heavily than a false alarm (assumed 5:1 ratio, since the dataset lacks loan amount/LGD data needed for a real cost figure).

**Key results**

| Model | AUC |
|---|---|
| Logistic regression (raw) | 0.714 |
| Logistic regression (capped utilization) | 0.811 |
| Logistic regression (+ DebtRatio/income fix) | 0.812 |
| Logistic regression (+ late-payment sentinel fix) | **0.856** |
| XGBoost (raw, default hyperparameters) | **0.860** |

**Limitations / honest scope**: this project produces a PD (probability of default) model only. A real credit decision requires $EL = PD \times LGD \times EAD$, and this dataset has no loan amount or loss-given-default data — so threshold/cost analysis here is illustrative of the *methodology*, not a deployable dollar-optimal cutoff. Not done: hyperparameter tuning, cross-validation (single stratified split used throughout), XGBoost wrapped in its own pipeline.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_scores = cross_val_score(logreg_pipeline, X, y, cv=cv, scoring='roc_auc')

print('fold AUCs:', cv_scores)
print(f'mean AUC: {cv_scores.mean():.4f}')
print(f'std AUC:  {cv_scores.std():.4f}')

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

xgb_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('model', XGBClassifier(random_state=42))
])

param_distributions = {
    'model__n_estimators': randint(100, 500),
    'model__max_depth': randint(3, 8),
    'model__learning_rate': uniform(0.01, 0.29),   # samples in [0.01, 0.30]
    'model__subsample': uniform(0.6, 0.4),          # samples in [0.6, 1.0]
    'model__colsample_bytree': uniform(0.6, 0.4),   # samples in [0.6, 1.0]
    'model__min_child_weight': randint(1, 8),
}

search = RandomizedSearchCV(
    xgb_pipeline,
    param_distributions=param_distributions,
    n_iter=25,
    cv=cv,
    scoring='roc_auc',
    random_state=42,
    n_jobs=-1
)

search.fit(X, y)

print('best params:', search.best_params_)
print('best CV AUC:', search.best_score_)